# Resume Data Cleaning

Clean and standardize resume data with predefined IT skills categorization

In [1]:
import json
import pandas as pd
import numpy as np
from collections import Counter
import re
from difflib import SequenceMatcher

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# Load the parsed resume data
with open('parsed_resumes.json', 'r') as f:
    data = json.load(f)

print(f"Loaded {len(data)} resumes")

# Count successfully parsed resumes
parsed_count = sum(1 for resume in data if resume.get('parsed') is not None)
print(f"Successfully parsed: {parsed_count} ({parsed_count/len(data)*100:.1f}%)")

Loaded 5029 resumes
Successfully parsed: 5029 (100.0%)


In [3]:
# Predefined IT skills by categories
IT_SKILLS = {
    'Programming Languages': [
        'Python', 'Java', 'JavaScript', 'TypeScript', 'C++', 'C#', 'C', 'Go', 'Rust',
        'Ruby', 'PHP', 'Swift', 'Kotlin', 'Scala', 'R', 'MATLAB', 'Perl', 'Dart',
        'Objective-C', 'Visual Basic', 'Assembly', 'Haskell', 'Clojure', 'F#',
        'Groovy', 'Lua', 'Julia', 'Erlang', 'Elixir'
    ],
    'Web Technologies': [
        'HTML', 'CSS', 'React', 'Angular', 'Vue.js', 'Node.js', 'Express.js',
        'Django', 'Flask', 'Spring', 'Laravel', 'CodeIgniter', 'Symfony',
        'ASP.NET', 'jQuery', 'Bootstrap', 'Sass', 'Less', 'Webpack', 'Gulp',
        'Grunt', 'Ember.js', 'Backbone.js', 'Svelte', 'Next.js', 'Nuxt.js',
        'Gatsby', 'GraphQL', 'REST API', 'SOAP', 'XML', 'JSON'
    ],
    'Databases': [
        'MySQL', 'PostgreSQL', 'MongoDB', 'SQLite', 'Oracle', 'SQL Server',
        'Redis', 'Cassandra', 'DynamoDB', 'CouchDB', 'Neo4j', 'Firebase',
        'MariaDB', 'Amazon RDS', 'Elasticsearch', 'InfluxDB', 'TimescaleDB',
        'SQL', 'NoSQL', 'Database Design', 'Database Administration'
    ],
    'Cloud & DevOps': [
        'AWS', 'Azure', 'Google Cloud', 'Docker', 'Kubernetes', 'Jenkins',
        'GitLab CI', 'GitHub Actions', 'Terraform', 'Ansible', 'Chef', 'Puppet',
        'Vagrant', 'CI/CD', 'DevOps', 'Microservices', 'Serverless',
        'Lambda', 'EC2', 'S3', 'CloudFormation', 'Helm', 'Istio',
        'Prometheus', 'Grafana', 'ELK Stack', 'Nginx', 'Apache'
    ],
    'Data Science & AI': [
        'Machine Learning', 'Deep Learning', 'Artificial Intelligence',
        'Data Science', 'Data Analysis', 'Statistics', 'Pandas', 'NumPy',
        'Scikit-learn', 'TensorFlow', 'PyTorch', 'Keras', 'OpenCV',
        'Natural Language Processing', 'Computer Vision', 'Big Data',
        'Apache Spark', 'Hadoop', 'Kafka', 'Tableau', 'Power BI',
        'Jupyter', 'R Studio', 'SPSS', 'SAS', 'Neural Networks'
    ],
    'Mobile Development': [
        'Android', 'iOS', 'React Native', 'Flutter', 'Xamarin', 'Ionic',
        'Cordova', 'PhoneGap', 'Swift', 'Kotlin', 'Objective-C',
        'Mobile Development', 'App Development'
    ],
    'Testing & QA': [
        'Unit Testing', 'Integration Testing', 'Test Automation', 'Selenium',
        'Jest', 'Mocha', 'Cypress', 'TestNG', 'JUnit', 'PyTest',
        'Quality Assurance', 'Manual Testing', 'Performance Testing',
        'Load Testing', 'Security Testing', 'API Testing', 'Postman'
    ],
    'Version Control & Tools': [
        'Git', 'GitHub', 'GitLab', 'Bitbucket', 'SVN', 'Mercurial',
        'JIRA', 'Confluence', 'Slack', 'Trello', 'Asana', 'VS Code',
        'IntelliJ IDEA', 'Eclipse', 'Visual Studio', 'Sublime Text',
        'Atom', 'Vim', 'Emacs'
    ],
    'Operating Systems': [
        'Linux', 'Ubuntu', 'CentOS', 'Red Hat', 'Debian', 'Windows',
        'macOS', 'Unix', 'FreeBSD', 'Windows Server', 'System Administration'
    ],
    'Security': [
        'Cybersecurity', 'Information Security', 'Network Security',
        'Penetration Testing', 'Ethical Hacking', 'Cryptography',
        'SSL/TLS', 'OAuth', 'JWT', 'OWASP', 'Firewall', 'VPN',
        'Security Audit', 'Vulnerability Assessment'
    ],
    'Networking': [
        'TCP/IP', 'HTTP/HTTPS', 'DNS', 'DHCP', 'Network Administration',
        'Routing', 'Switching', 'Load Balancing', 'CDN', 'VPN',
        'Network Security', 'Cisco', 'Juniper'
    ],
    'Business Intelligence': [
        'Business Intelligence', 'Data Warehousing', 'ETL', 'OLAP',
        'Data Mining', 'Reporting', 'Analytics', 'KPI', 'Dashboards',
        'Business Analysis', 'Requirements Analysis'
    ]
}

# Predefined Soft Skills by categories
SOFT_SKILLS = {
    'Communication': [
        'Communication', 'Verbal Communication', 'Written Communication', 'Presentation Skills',
        'Public Speaking', 'Active Listening', 'Storytelling', 'Negotiation',
        'Interpersonal Skills', 'Cross-cultural Communication', 'Business Writing',
        'Technical Writing', 'Documentation', 'Reporting', 'Email Etiquette'
    ],
    'Leadership & Management': [
        'Leadership', 'Team Leadership', 'Project Management', 'People Management',
        'Strategic Planning', 'Decision Making', 'Delegation', 'Mentoring',
        'Coaching', 'Conflict Resolution', 'Change Management', 'Risk Management',
        'Resource Management', 'Budget Management', 'Stakeholder Management'
    ],
    'Collaboration & Teamwork': [
        'Teamwork', 'Collaboration', 'Team Building', 'Cross-functional Collaboration',
        'Relationship Building', 'Networking', 'Partnership Development',
        'Consensus Building', 'Team Player', 'Cooperative', 'Supportive'
    ],
    'Problem Solving & Critical Thinking': [
        'Problem Solving', 'Critical Thinking', 'Analytical Thinking', 'Logical Thinking',
        'Creative Thinking', 'Innovation', 'Troubleshooting', 'Root Cause Analysis',
        'Decision Analysis', 'Strategic Thinking', 'Systems Thinking',
        'Research Skills', 'Data-driven Decision Making'
    ],
    'Personal Effectiveness': [
        'Time Management', 'Organization', 'Prioritization', 'Multi-tasking',
        'Self-motivated', 'Initiative', 'Proactive', 'Detail-oriented',
        'Results-oriented', 'Goal-oriented', 'Deadline Management',
        'Work Independently', 'Self-starter', 'Productivity', 'Efficiency'
    ],
    'Adaptability & Learning': [
        'Adaptability', 'Flexibility', 'Quick Learner', 'Continuous Learning',
        'Growth Mindset', 'Resilience', 'Open-minded', 'Agile', 'Versatile',
        'Change Adaptation', 'Learning Agility', 'Curiosity', 'Self-improvement'
    ],
    'Emotional Intelligence': [
        'Emotional Intelligence', 'Empathy', 'Self-awareness', 'Social Awareness',
        'Emotional Regulation', 'Stress Management', 'Patience', 'Compassion',
        'Understanding', 'Interpersonal Awareness', 'Cultural Sensitivity'
    ],
    'Professional & Ethics': [
        'Professional', 'Work Ethic', 'Integrity', 'Accountability', 'Responsibility',
        'Reliability', 'Dependable', 'Trustworthy', 'Ethical', 'Confidentiality',
        'Commitment', 'Dedication', 'Punctual', 'Honest', 'Transparent'
    ],
    'Customer & Client Focus': [
        'Customer Service', 'Client Relations', 'Customer Focus', 'Client Management',
        'Customer Success', 'Service-oriented', 'Customer Satisfaction',
        'Client Communication', 'Account Management', 'Customer Experience',
        'Complaint Resolution', 'Customer Support'
    ],
    'Creativity & Innovation': [
        'Creative', 'Innovative', 'Imagination', 'Brainstorming', 'Ideation',
        'Design Thinking', 'Out-of-the-box Thinking', 'Conceptualization',
        'Artistic', 'Inventive', 'Original Thinking', 'Entrepreneurial'
    ]
}

# Predefined Language Skills
LANGUAGE_SKILLS = {
    'English Languages': [
        'English', 'Business English', 'Technical English', 'English Proficiency',
        'English Writing', 'English Speaking', 'English Communication'
    ],
    'Asian Languages': [
        'Hindi', 'Mandarin', 'Chinese', 'Japanese', 'Korean', 'Tamil', 'Telugu',
        'Kannada', 'Malayalam', 'Bengali', 'Gujarati', 'Marathi', 'Punjabi',
        'Urdu', 'Sanskrit', 'Thai', 'Vietnamese', 'Indonesian', 'Malay',
        'Tagalog', 'Filipino'
    ],
    'European Languages': [
        'Spanish', 'French', 'German', 'Italian', 'Portuguese', 'Dutch',
        'Russian', 'Polish', 'Swedish', 'Norwegian', 'Danish', 'Finnish',
        'Greek', 'Turkish', 'Czech', 'Hungarian', 'Romanian', 'Bulgarian',
        'Croatian', 'Serbian', 'Ukrainian'
    ],
    'Middle Eastern Languages': [
        'Arabic', 'Hebrew', 'Persian', 'Farsi', 'Kurdish', 'Pashto'
    ],
    'African Languages': [
        'Swahili', 'Zulu', 'Yoruba', 'Hausa', 'Amharic', 'Somali'
    ],
    'Sign Languages': [
        'Sign Language', 'ASL', 'American Sign Language', 'BSL', 
        'British Sign Language', 'ISL', 'Indian Sign Language'
    ],
    'Language Proficiency Levels': [
        'Native', 'Fluent', 'Proficient', 'Intermediate', 'Conversational',
        'Basic', 'Beginner', 'A1', 'A2', 'B1', 'B2', 'C1', 'C2',
        'TOEFL', 'IELTS', 'TOEIC', 'Cambridge English'
    ]
}

# Create flat lists for matching - IT Skills
all_predefined_skills = []
skill_to_category = {}

for category, skills in IT_SKILLS.items():
    for skill in skills:
        all_predefined_skills.append(skill.lower())
        skill_to_category[skill.lower()] = {
            'standardized_name': skill,
            'category': category
        }

# Create flat lists for matching - Soft Skills
all_predefined_soft_skills = []
soft_skill_to_category = {}

for category, skills in SOFT_SKILLS.items():
    for skill in skills:
        all_predefined_soft_skills.append(skill.lower())
        soft_skill_to_category[skill.lower()] = {
            'standardized_name': skill,
            'category': category
        }

# Create flat lists for matching - Language Skills
all_predefined_languages = []
language_to_category = {}

for category, languages in LANGUAGE_SKILLS.items():
    for language in languages:
        all_predefined_languages.append(language.lower())
        language_to_category[language.lower()] = {
            'standardized_name': language,
            'category': category
        }

print(f"Total predefined IT skills: {len(all_predefined_skills)}")
print(f"Total predefined soft skills: {len(all_predefined_soft_skills)}")
print(f"Total predefined language skills: {len(all_predefined_languages)}")
print(f"\nIT Skill categories: {list(IT_SKILLS.keys())}")
print(f"Soft Skill categories: {list(SOFT_SKILLS.keys())}")
print(f"Language categories: {list(LANGUAGE_SKILLS.keys())}")

Total predefined IT skills: 234
Total predefined soft skills: 132
Total predefined language skills: 85

IT Skill categories: ['Programming Languages', 'Web Technologies', 'Databases', 'Cloud & DevOps', 'Data Science & AI', 'Mobile Development', 'Testing & QA', 'Version Control & Tools', 'Operating Systems', 'Security', 'Networking', 'Business Intelligence']
Soft Skill categories: ['Communication', 'Leadership & Management', 'Collaboration & Teamwork', 'Problem Solving & Critical Thinking', 'Personal Effectiveness', 'Adaptability & Learning', 'Emotional Intelligence', 'Professional & Ethics', 'Customer & Client Focus', 'Creativity & Innovation']
Language categories: ['English Languages', 'Asian Languages', 'European Languages', 'Middle Eastern Languages', 'African Languages', 'Sign Languages', 'Language Proficiency Levels']


In [4]:
def similarity(a, b):
    """Calculate similarity between two strings"""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def find_best_skill_match(user_skill, threshold=0.8):
    """Find the best matching predefined IT skill for a user skill"""
    user_skill_clean = re.sub(r'[^a-zA-Z0-9\s\.#+]', '', user_skill.lower().strip())
    
    if not user_skill_clean or len(user_skill_clean) < 2:
        return None
    
    # Skip very short skills that are likely false positives
    if len(user_skill_clean) <= 2 and user_skill_clean not in ['ai', 'ml', 'ui', 'ux', 'qa', 'ci', 'cd']:
        return None
    
    # Direct match first
    if user_skill_clean in all_predefined_skills:
        return skill_to_category[user_skill_clean]
    
    # Partial matches
    best_match = None
    best_score = 0
    
    for predefined_skill in all_predefined_skills:
        # Skip single character matches unless they're exact matches
        if len(predefined_skill) == 1 and predefined_skill != user_skill_clean:
            continue
            
        # Check if user skill contains the predefined skill or vice versa
        if predefined_skill in user_skill_clean or user_skill_clean in predefined_skill:
            score = max(len(predefined_skill) / len(user_skill_clean), 
                       len(user_skill_clean) / len(predefined_skill))
            if score > best_score and score >= threshold:
                best_score = score
                best_match = skill_to_category[predefined_skill]
        
        # Similarity matching
        sim_score = similarity(user_skill_clean, predefined_skill)
        if sim_score > best_score and sim_score >= threshold:
            best_score = sim_score
            best_match = skill_to_category[predefined_skill]
    
    return best_match

def find_best_soft_skill_match(user_skill, threshold=0.75):
    """Find the best matching predefined soft skill for a user skill"""
    user_skill_clean = re.sub(r'[^a-zA-Z0-9\s\-]', '', user_skill.lower().strip())
    
    if not user_skill_clean or len(user_skill_clean) < 3:  # Increased minimum length
        return None
    
    # Direct match first
    if user_skill_clean in all_predefined_soft_skills:
        return soft_skill_to_category[user_skill_clean]
    
    # Partial matches
    best_match = None
    best_score = 0
    
    for predefined_skill in all_predefined_soft_skills:
        # For soft skills, require better word boundary matching
        words_in_skill = predefined_skill.split()
        words_in_user = user_skill_clean.split()
        
        # Check for whole word matches
        if len(words_in_skill) == 1 and len(words_in_user) == 1:
            if predefined_skill == user_skill_clean:
                return soft_skill_to_category[predefined_skill]
        
        # Check if user skill contains the predefined skill or vice versa
        if predefined_skill in user_skill_clean or user_skill_clean in predefined_skill:
            score = max(len(predefined_skill) / len(user_skill_clean), 
                       len(user_skill_clean) / len(predefined_skill))
            if score > best_score and score >= threshold:
                best_score = score
                best_match = soft_skill_to_category[predefined_skill]
        
        # Similarity matching
        sim_score = similarity(user_skill_clean, predefined_skill)
        if sim_score > best_score and sim_score >= threshold:
            best_score = sim_score
            best_match = soft_skill_to_category[predefined_skill]
    
    return best_match

def find_best_language_match(user_skill, threshold=0.85):
    """Find the best matching predefined language for a user skill"""
    user_skill_clean = re.sub(r'[^a-zA-Z0-9\s]', '', user_skill.lower().strip())
    
    if not user_skill_clean or len(user_skill_clean) < 3:  # Increased minimum length
        return None
    
    # Direct match first
    if user_skill_clean in all_predefined_languages:
        return language_to_category[user_skill_clean]
    
    # Partial matches - more strict for languages
    best_match = None
    best_score = 0
    
    for predefined_lang in all_predefined_languages:
        # Check exact word match (for multi-word languages)
        if predefined_lang == user_skill_clean:
            return language_to_category[predefined_lang]
        
        # Check if the language name is contained as a whole word
        if f" {predefined_lang} " in f" {user_skill_clean} " or \
           user_skill_clean.startswith(f"{predefined_lang} ") or \
           user_skill_clean.endswith(f" {predefined_lang}"):
            score = 1.0
            if score > best_score:
                best_score = score
                best_match = language_to_category[predefined_lang]
        
        # Similarity matching with higher threshold for languages
        sim_score = similarity(user_skill_clean, predefined_lang)
        if sim_score > best_score and sim_score >= threshold:
            best_score = sim_score
            best_match = language_to_category[predefined_lang]
    
    return best_match

def clean_skills(skills_list):
    """Clean and standardize a list of skills, separating IT, soft, and language skills"""
    if not isinstance(skills_list, list):
        return [], [], [], [], [], []
    
    cleaned_it_skills = []
    it_skill_categories = []
    cleaned_soft_skills = []
    soft_skill_categories = []
    cleaned_languages = []
    language_categories = []
    
    for skill in skills_list:
        if not skill or not isinstance(skill, str):
            continue
            
        # Skip very short or very long skills (likely noise)
        skill_stripped = skill.strip()
        if len(skill_stripped) < 2 or len(skill_stripped) > 50:
            continue
        
        # Skip obvious noise patterns
        if skill_stripped.lower() in ['ltd', 'inc', 'com', 'www', 'http', 'https', 'email', 'e-mail', 'mail']:
            continue
            
        # Skip single characters except known abbreviations
        if len(skill_stripped) == 1 and skill_stripped.lower() not in ['r', 'c']:
            continue
        
        # Try to match language first (as they are often misclassified)
        language_match = find_best_language_match(skill)
        if language_match:
            cleaned_languages.append(language_match['standardized_name'])
            language_categories.append(language_match['category'])
        else:
            # Try to match IT skill
            it_match = find_best_skill_match(skill)
            if it_match:
                cleaned_it_skills.append(it_match['standardized_name'])
                it_skill_categories.append(it_match['category'])
            else:
                # Try to match soft skill
                soft_match = find_best_soft_skill_match(skill)
                if soft_match:
                    cleaned_soft_skills.append(soft_match['standardized_name'])
                    soft_skill_categories.append(soft_match['category'])
    
    # Remove duplicates while preserving order
    seen_it = set()
    unique_it_skills = []
    unique_it_categories = []
    
    for skill, category in zip(cleaned_it_skills, it_skill_categories):
        if skill not in seen_it:
            seen_it.add(skill)
            unique_it_skills.append(skill)
            unique_it_categories.append(category)
    
    seen_soft = set()
    unique_soft_skills = []
    unique_soft_categories = []
    
    for skill, category in zip(cleaned_soft_skills, soft_skill_categories):
        if skill not in seen_soft:
            seen_soft.add(skill)
            unique_soft_skills.append(skill)
            unique_soft_categories.append(category)
    
    seen_lang = set()
    unique_languages = []
    unique_language_categories = []
    
    for lang, category in zip(cleaned_languages, language_categories):
        if lang not in seen_lang:
            seen_lang.add(lang)
            unique_languages.append(lang)
            unique_language_categories.append(category)
    
    return (unique_it_skills, unique_it_categories, 
            unique_soft_skills, unique_soft_categories,
            unique_languages, unique_language_categories)

# Test the cleaning function with problematic examples
test_skills = ['python', 'JAVA', 'react.js', 'machine learning', 'nodejs', 'html5', 'css3', 
               'leadership', 'communication', 'team player', 'problem solving', 'time management',
               'english', 'hindi', 'spanish', 'mandarin chinese', 'french',
               'offers', 'contract', 'excel', 'records', 'credit', 'r', 'c', 'reporting']
it_skills, it_categories, soft_skills, soft_categories, languages, lang_categories = clean_skills(test_skills)
print(f"Test input: {test_skills}")
print(f"\nIT Skills: {it_skills}")
print(f"IT Categories: {it_categories}")
print(f"\nSoft Skills: {soft_skills}")
print(f"Soft Categories: {soft_categories}")
print(f"\nLanguages: {languages}")
print(f"Language Categories: {lang_categories}")

# Test individual problematic matches
print(f"\nTesting individual matches:")
print(f"'offers' -> IT: {find_best_skill_match('offers')}, Soft: {find_best_soft_skill_match('offers')}")
print(f"'contract' -> IT: {find_best_skill_match('contract')}, Soft: {find_best_soft_skill_match('contract')}")
print(f"'records' -> IT: {find_best_skill_match('records')}, Soft: {find_best_soft_skill_match('records')}")
print(f"'credit' -> IT: {find_best_skill_match('credit')}, Soft: {find_best_soft_skill_match('credit')}")
print(f"'r' -> IT: {find_best_skill_match('r')}, Soft: {find_best_soft_skill_match('r')}")
print(f"'c' -> IT: {find_best_skill_match('c')}, Soft: {find_best_soft_skill_match('c')}")

Test input: ['python', 'JAVA', 'react.js', 'machine learning', 'nodejs', 'html5', 'css3', 'leadership', 'communication', 'team player', 'problem solving', 'time management', 'english', 'hindi', 'spanish', 'mandarin chinese', 'french', 'offers', 'contract', 'excel', 'records', 'credit', 'r', 'c', 'reporting']

IT Skills: ['Python', 'Java', 'React', 'Machine Learning', 'Node.js', 'HTML', 'S3', 'Reporting']
IT Categories: ['Programming Languages', 'Programming Languages', 'Web Technologies', 'Data Science & AI', 'Web Technologies', 'Web Technologies', 'Cloud & DevOps', 'Business Intelligence']

Soft Skills: ['Leadership', 'Communication', 'Team Player', 'Problem Solving', 'Time Management']
Soft Categories: ['Leadership & Management', 'Communication', 'Collaboration & Teamwork', 'Problem Solving & Critical Thinking', 'Personal Effectiveness']

Languages: ['English', 'Hindi', 'Spanish', 'Mandarin', 'French']
Language Categories: ['English Languages', 'Asian Languages', 'European Languages'

In [5]:
# Clean all resume skills (IT, soft, and language skills)
print("Cleaning skills for all resumes...")

original_skills_count = 0
cleaned_it_skills_count = 0
cleaned_soft_skills_count = 0
cleaned_languages_count = 0
unmatched_skills = Counter()

for i, resume in enumerate(data):
    if resume.get('parsed') and 'skills' in resume['parsed']:
        original_skills = resume['parsed']['skills']
        
        if isinstance(original_skills, list):
            original_skills_count += len(original_skills)
            
            # Clean skills (now separating IT, soft, and language skills)
            it_skills, it_categories, soft_skills, soft_categories, languages, lang_categories = clean_skills(original_skills)
            cleaned_it_skills_count += len(it_skills)
            cleaned_soft_skills_count += len(soft_skills)
            cleaned_languages_count += len(languages)
            
            # Track unmatched skills
            for skill in original_skills:
                if isinstance(skill, str) and len(skill.strip()) >= 2:
                    it_match = find_best_skill_match(skill)
                    soft_match = find_best_soft_skill_match(skill)
                    lang_match = find_best_language_match(skill)
                    if not it_match and not soft_match and not lang_match:
                        unmatched_skills[skill.lower().strip()] += 1
            
            # Update the resume data
            resume['parsed']['skills_original'] = original_skills
            resume['parsed']['it_skills'] = it_skills
            resume['parsed']['it_skill_categories'] = it_categories
            resume['parsed']['soft_skills'] = soft_skills
            resume['parsed']['soft_skill_categories'] = soft_categories
            resume['parsed']['languages'] = languages
            resume['parsed']['language_categories'] = lang_categories
            # Keep IT skills as main skills for backward compatibility
            resume['parsed']['skills'] = it_skills
    
    if (i + 1) % 1000 == 0:
        print(f"Processed {i + 1}/{len(data)} resumes")

print(f"\nSkill cleaning completed!")
print(f"Original skills: {original_skills_count}")
print(f"Cleaned IT skills: {cleaned_it_skills_count}")
print(f"Cleaned soft skills: {cleaned_soft_skills_count}")
print(f"Cleaned language skills: {cleaned_languages_count}")
print(f"Total cleaned skills: {cleaned_it_skills_count + cleaned_soft_skills_count + cleaned_languages_count}")
print(f"IT skill retention rate: {cleaned_it_skills_count/original_skills_count*100:.1f}%")
print(f"Soft skill retention rate: {cleaned_soft_skills_count/original_skills_count*100:.1f}%")
print(f"Language skill retention rate: {cleaned_languages_count/original_skills_count*100:.1f}%")
print(f"Total retention rate: {(cleaned_it_skills_count + cleaned_soft_skills_count + cleaned_languages_count)/original_skills_count*100:.1f}%")
print(f"\nTop 15 unmatched skills:")
for skill, count in unmatched_skills.most_common(15):
    print(f"  {skill}: {count}")

Cleaning skills for all resumes...
Processed 1000/5029 resumes
Processed 2000/5029 resumes
Processed 3000/5029 resumes
Processed 4000/5029 resumes
Processed 5000/5029 resumes

Skill cleaning completed!
Original skills: 169711
Cleaned IT skills: 13350
Cleaned soft skills: 13093
Cleaned language skills: 8343
Total cleaned skills: 34786
IT skill retention rate: 7.9%
Soft skill retention rate: 7.7%
Language skill retention rate: 4.9%
Total retention rate: 20.5%

Top 15 unmatched skills:
  birth: 1249
  status: 1090
  office: 913
  ltd: 848
  excel: 719
  training: 692
  technology: 637
  maintenance: 582
  e-mail: 519
  passport: 511
  hobbies: 465
  word: 461
  construction: 445
  it: 441
  materials: 435


In [6]:
# Generate cleaning report
def generate_cleaning_report():
    """Generate a comprehensive cleaning report including IT, soft, and language skills"""
    
    # IT Skills analysis
    all_cleaned_it_skills = []
    all_it_categories = []
    it_skills_per_resume = []
    
    # Soft Skills analysis
    all_cleaned_soft_skills = []
    all_soft_categories = []
    soft_skills_per_resume = []
    
    # Language Skills analysis
    all_cleaned_languages = []
    all_language_categories = []
    languages_per_resume = []
    
    # Other metrics
    experience_values = []
    companies_count = []
    universities_count = []
    
    for resume in data:
        if resume.get('parsed'):
            parsed = resume['parsed']
            
            # IT Skills
            if 'it_skills' in parsed and isinstance(parsed['it_skills'], list):
                it_skills = parsed['it_skills']
                all_cleaned_it_skills.extend(it_skills)
                it_skills_per_resume.append(len(it_skills))
                
                if 'it_skill_categories' in parsed:
                    all_it_categories.extend(parsed['it_skill_categories'])
            elif 'skills' in parsed and isinstance(parsed['skills'], list):
                # Fallback for backward compatibility
                skills = parsed['skills']
                all_cleaned_it_skills.extend(skills)
                it_skills_per_resume.append(len(skills))
                
                if 'skill_categories' in parsed:
                    all_it_categories.extend(parsed['skill_categories'])
            else:
                it_skills_per_resume.append(0)
            
            # Soft Skills
            if 'soft_skills' in parsed and isinstance(parsed['soft_skills'], list):
                soft_skills = parsed['soft_skills']
                all_cleaned_soft_skills.extend(soft_skills)
                soft_skills_per_resume.append(len(soft_skills))
                
                if 'soft_skill_categories' in parsed:
                    all_soft_categories.extend(parsed['soft_skill_categories'])
            else:
                soft_skills_per_resume.append(0)
            
            # Language Skills
            if 'languages' in parsed and isinstance(parsed['languages'], list):
                languages = parsed['languages']
                all_cleaned_languages.extend(languages)
                languages_per_resume.append(len(languages))
                
                if 'language_categories' in parsed:
                    all_language_categories.extend(parsed['language_categories'])
            else:
                languages_per_resume.append(0)
            
            # Experience
            if 'total_exp' in parsed:
                experience_values.append(parsed['total_exp'])
            
            # Companies
            if 'Companies worked at' in parsed and isinstance(parsed['Companies worked at'], list):
                companies_count.append(len(parsed['Companies worked at']))
            
            # Universities
            if 'university' in parsed and isinstance(parsed['university'], list):
                universities_count.append(len(parsed['university']))
    
    # Calculate total skills per resume
    total_skills_per_resume = [it + soft + lang for it, soft, lang in 
                               zip(it_skills_per_resume, soft_skills_per_resume, languages_per_resume)]
    
    # Generate report
    report = {
        "total_resumes": len(data),
        "parsed_resumes": sum(1 for r in data if r.get('parsed')),
        "it_skills_stats": {
            "total_unique_skills": len(set(all_cleaned_it_skills)),
            "total_skill_mentions": len(all_cleaned_it_skills),
            "avg_skills_per_resume": np.mean(it_skills_per_resume) if it_skills_per_resume else 0,
            "top_skills": dict(Counter(all_cleaned_it_skills).most_common(10)),
            "skills_by_category": dict(Counter(all_it_categories))
        },
        "soft_skills_stats": {
            "total_unique_skills": len(set(all_cleaned_soft_skills)),
            "total_skill_mentions": len(all_cleaned_soft_skills),
            "avg_skills_per_resume": np.mean(soft_skills_per_resume) if soft_skills_per_resume else 0,
            "top_skills": dict(Counter(all_cleaned_soft_skills).most_common(10)),
            "skills_by_category": dict(Counter(all_soft_categories))
        },
        "language_skills_stats": {
            "total_unique_languages": len(set(all_cleaned_languages)),
            "total_language_mentions": len(all_cleaned_languages),
            "avg_languages_per_resume": np.mean(languages_per_resume) if languages_per_resume else 0,
            "top_languages": dict(Counter(all_cleaned_languages).most_common(10)),
            "languages_by_category": dict(Counter(all_language_categories))
        },
        "combined_skills_stats": {
            "total_unique_skills": len(set(all_cleaned_it_skills + all_cleaned_soft_skills + all_cleaned_languages)),
            "total_skill_mentions": len(all_cleaned_it_skills) + len(all_cleaned_soft_skills) + len(all_cleaned_languages),
            "avg_total_skills_per_resume": np.mean(total_skills_per_resume) if total_skills_per_resume else 0
        },
        "experience_stats": {
            "avg_experience": np.mean(experience_values) if experience_values else 0,
            "median_experience": np.median(experience_values) if experience_values else 0,
            "max_experience": max(experience_values) if experience_values else 0,
            "resumes_with_experience": sum(1 for exp in experience_values if exp > 0)
        },
        "companies_stats": {
            "avg_companies_per_resume": np.mean(companies_count) if companies_count else 0,
            "resumes_with_companies": sum(1 for c in companies_count if c > 0)
        },
        "education_stats": {
            "avg_universities_per_resume": np.mean(universities_count) if universities_count else 0,
            "resumes_with_university": sum(1 for u in universities_count if u > 0)
        }
    }
    
    return report

# Generate and display report
report = generate_cleaning_report()

print("Data Cleaning Report with IT, Soft, and Language Skills")
print("=" * 60)
print(f"Total Resumes: {report['total_resumes']}")
print(f"Parsed Resumes: {report['parsed_resumes']}")

print(f"\n{'='*60}")
print("IT SKILLS STATISTICS")
print(f"{'='*60}")
print(f"  Total Unique IT Skills: {report['it_skills_stats']['total_unique_skills']}")
print(f"  Total IT Skill Mentions: {report['it_skills_stats']['total_skill_mentions']}")
print(f"  Avg IT Skills per Resume: {report['it_skills_stats']['avg_skills_per_resume']:.2f}")

print(f"\nTop 10 IT Skills:")
for skill, count in report['it_skills_stats']['top_skills'].items():
    print(f"  {skill}: {count}")

print(f"\nIT Skills by Category:")
for category, count in sorted(report['it_skills_stats']['skills_by_category'].items(), 
                             key=lambda x: x[1], reverse=True):
    print(f"  {category}: {count}")

print(f"\n{'='*60}")
print("SOFT SKILLS STATISTICS")
print(f"{'='*60}")
print(f"  Total Unique Soft Skills: {report['soft_skills_stats']['total_unique_skills']}")
print(f"  Total Soft Skill Mentions: {report['soft_skills_stats']['total_skill_mentions']}")
print(f"  Avg Soft Skills per Resume: {report['soft_skills_stats']['avg_skills_per_resume']:.2f}")

print(f"\nTop 10 Soft Skills:")
for skill, count in report['soft_skills_stats']['top_skills'].items():
    print(f"  {skill}: {count}")

print(f"\nSoft Skills by Category:")
for category, count in sorted(report['soft_skills_stats']['skills_by_category'].items(), 
                             key=lambda x: x[1], reverse=True):
    print(f"  {category}: {count}")

print(f"\n{'='*60}")
print("LANGUAGE SKILLS STATISTICS")
print(f"{'='*60}")
print(f"  Total Unique Languages: {report['language_skills_stats']['total_unique_languages']}")
print(f"  Total Language Mentions: {report['language_skills_stats']['total_language_mentions']}")
print(f"  Avg Languages per Resume: {report['language_skills_stats']['avg_languages_per_resume']:.2f}")

print(f"\nTop 10 Languages:")
for lang, count in report['language_skills_stats']['top_languages'].items():
    print(f"  {lang}: {count}")

print(f"\nLanguages by Category:")
for category, count in sorted(report['language_skills_stats']['languages_by_category'].items(), 
                             key=lambda x: x[1], reverse=True):
    print(f"  {category}: {count}")

print(f"\n{'='*60}")
print("COMBINED SKILLS SUMMARY")
print(f"{'='*60}")
print(f"  Total Unique Skills (IT + Soft + Languages): {report['combined_skills_stats']['total_unique_skills']}")
print(f"  Total Skill Mentions (IT + Soft + Languages): {report['combined_skills_stats']['total_skill_mentions']}")
print(f"  Avg Total Skills per Resume: {report['combined_skills_stats']['avg_total_skills_per_resume']:.2f}")

print(f"\n{'='*60}")
print("OTHER STATISTICS")
print(f"{'='*60}")
print(f"Experience:")
print(f"  Average Experience: {report['experience_stats']['avg_experience']:.2f} years")
print(f"  Median Experience: {report['experience_stats']['median_experience']:.2f} years")
print(f"  Resumes with Experience: {report['experience_stats']['resumes_with_experience']}")

print(f"\nCompanies:")
print(f"  Avg Companies per Resume: {report['companies_stats']['avg_companies_per_resume']:.2f}")
print(f"  Resumes with Companies: {report['companies_stats']['resumes_with_companies']}")

print(f"\nEducation:")
print(f"  Avg Universities per Resume: {report['education_stats']['avg_universities_per_resume']:.2f}")
print(f"  Resumes with University: {report['education_stats']['resumes_with_university']}")

Data Cleaning Report with IT, Soft, and Language Skills
Total Resumes: 5029
Parsed Resumes: 5029

IT SKILLS STATISTICS
  Total Unique IT Skills: 201
  Total IT Skill Mentions: 13350
  Avg IT Skills per Resume: 2.65

Top 10 IT Skills:
  Go: 881
  Windows: 670
  Mobile Development: 651
  Machine Learning: 479
  Database Design: 460
  Natural Language Processing: 421
  Database Administration: 381
  Integration Testing: 351
  Reporting: 332
  Backbone.js: 331

IT Skills by Category:
  Data Science & AI: 2250
  Programming Languages: 2176
  Databases: 1807
  Web Technologies: 1382
  Operating Systems: 1123
  Mobile Development: 975
  Business Intelligence: 878
  Testing & QA: 871
  Networking: 662
  Security: 520
  Cloud & DevOps: 416
  Version Control & Tools: 290

SOFT SKILLS STATISTICS
  Total Unique Soft Skills: 124
  Total Soft Skill Mentions: 13093
  Avg Soft Skills per Resume: 2.60

Top 10 Soft Skills:
  Project Management: 1012
  Organization: 998
  Delegation: 933
  Customer Satis

In [7]:
# Generate cleaning report
def generate_cleaning_report():
    """Generate a comprehensive cleaning report"""
    
    # Skills analysis
    all_cleaned_skills = []
    all_categories = []
    skills_per_resume = []
    
    # Other metrics
    experience_values = []
    companies_count = []
    universities_count = []
    
    for resume in data:
        if resume.get('parsed'):
            parsed = resume['parsed']
            
            # Skills
            if 'skills' in parsed and isinstance(parsed['skills'], list):
                skills = parsed['skills']
                all_cleaned_skills.extend(skills)
                skills_per_resume.append(len(skills))
                
                if 'skill_categories' in parsed:
                    all_categories.extend(parsed['skill_categories'])
            
            # Experience
            if 'total_exp' in parsed:
                experience_values.append(parsed['total_exp'])
            
            # Companies
            if 'Companies worked at' in parsed and isinstance(parsed['Companies worked at'], list):
                companies_count.append(len(parsed['Companies worked at']))
            
            # Universities
            if 'university' in parsed and isinstance(parsed['university'], list):
                universities_count.append(len(parsed['university']))
    
    # Generate report
    report = {
        "total_resumes": len(data),
        "parsed_resumes": sum(1 for r in data if r.get('parsed')),
        "skills_stats": {
            "total_unique_skills": len(set(all_cleaned_skills)),
            "total_skill_mentions": len(all_cleaned_skills),
            "avg_skills_per_resume": np.mean(skills_per_resume) if skills_per_resume else 0,
            "top_skills": dict(Counter(all_cleaned_skills).most_common(10)),
            "skills_by_category": dict(Counter(all_categories))
        },
        "experience_stats": {
            "avg_experience": np.mean(experience_values) if experience_values else 0,
            "median_experience": np.median(experience_values) if experience_values else 0,
            "max_experience": max(experience_values) if experience_values else 0,
            "resumes_with_experience": sum(1 for exp in experience_values if exp > 0)
        },
        "companies_stats": {
            "avg_companies_per_resume": np.mean(companies_count) if companies_count else 0,
            "resumes_with_companies": sum(1 for c in companies_count if c > 0)
        },
        "education_stats": {
            "avg_universities_per_resume": np.mean(universities_count) if universities_count else 0,
            "resumes_with_university": sum(1 for u in universities_count if u > 0)
        }
    }
    
    return report

# Generate and display report
report = generate_cleaning_report()

print("Data Cleaning Report")
print("=" * 50)
print(f"Total Resumes: {report['total_resumes']}")
print(f"Parsed Resumes: {report['parsed_resumes']}")

print(f"\nSkills Statistics:")
print(f"  Total Unique Skills: {report['skills_stats']['total_unique_skills']}")
print(f"  Total Skill Mentions: {report['skills_stats']['total_skill_mentions']}")
print(f"  Avg Skills per Resume: {report['skills_stats']['avg_skills_per_resume']:.2f}")

print(f"\nTop 10 Skills:")
for skill, count in report['skills_stats']['top_skills'].items():
    print(f"  {skill}: {count}")

print(f"\nSkills by Category:")
for category, count in sorted(report['skills_stats']['skills_by_category'].items(), 
                             key=lambda x: x[1], reverse=True):
    print(f"  {category}: {count}")

print(f"\nExperience Statistics:")
print(f"  Average Experience: {report['experience_stats']['avg_experience']:.2f} years")
print(f"  Median Experience: {report['experience_stats']['median_experience']:.2f} years")
print(f"  Resumes with Experience: {report['experience_stats']['resumes_with_experience']}")

print(f"\nOther Statistics:")
print(f"  Avg Companies per Resume: {report['companies_stats']['avg_companies_per_resume']:.2f}")
print(f"  Resumes with Companies: {report['companies_stats']['resumes_with_companies']}")
print(f"  Avg Universities per Resume: {report['education_stats']['avg_universities_per_resume']:.2f}")
print(f"  Resumes with University: {report['education_stats']['resumes_with_university']}")

Data Cleaning Report
Total Resumes: 5029
Parsed Resumes: 5029

Skills Statistics:
  Total Unique Skills: 201
  Total Skill Mentions: 13350
  Avg Skills per Resume: 2.65

Top 10 Skills:
  Go: 881
  Windows: 670
  Mobile Development: 651
  Machine Learning: 479
  Database Design: 460
  Natural Language Processing: 421
  Database Administration: 381
  Integration Testing: 351
  Reporting: 332
  Backbone.js: 331

Skills by Category:

Experience Statistics:
  Average Experience: 8.72 years
  Median Experience: 0.00 years
  Resumes with Experience: 2441

Other Statistics:
  Avg Companies per Resume: 62.04
  Resumes with Companies: 4982
  Avg Universities per Resume: 0.48
  Resumes with University: 1596


In [8]:
# Save cleaned data
with open('cleaned_resumes.json', 'w') as f:
    json.dump(data, f, indent=2)

# Save cleaning report
with open('cleaning_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print("\nCleaned data saved to 'cleaned_resumes.json'")
print("Cleaning report saved to 'cleaning_report.json'")
print("\nData cleaning completed successfully!")


Cleaned data saved to 'cleaned_resumes.json'
Cleaning report saved to 'cleaning_report.json'

Data cleaning completed successfully!
